# Full Pipeline: HL7 v2 → OMOP CDM

The supported two-hop route: HL7 v2 → FHIR R4 → OMOP, plus offline
concept normalization of the carried diagnosis codes.

> **Prerequisites** — `pip install "portiere-health[polars]"`. This notebook is
> **fully offline**: no model downloads, no network calls, no API keys. It uses
> the bundled ICD-10-CM / LOINC / RxNorm vocabulary subset and BM25 lexical
> retrieval, and asserts the no-egress guarantee with `offline=True`.
> Every cell is deterministic — run top to bottom.

In [1]:
import json
import tempfile
from pathlib import Path

import polars as pl

import portiere
from portiere import __version__

WORK = Path(tempfile.mkdtemp(prefix="portiere_nb_"))
print(f"portiere v{__version__} — working dir: {WORK}")

portiere v0.4.0 — working dir: /var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_qunsbv60


## The route: HL7 v2 → FHIR R4 → OMOP CDM

Portiere ships crossmaps for `hl7v2 → fhir_r4` and `fhir_r4 → omop`; chaining
them is the supported route (it mirrors real-world practice — v2 feeds are
usually landed as FHIR first). Each hop is explicit and auditable.

In [2]:
ADT_FEED = """MSH|^~\\&|HIS|GH|EHR|GH|20240115103000||ADT^A01|MSG0001|P|2.5.1
PID|1||MRN-10001||DOE^JANE||19850322|F
PV1|1|I||EMER|||||||||||||||VN-5001|||||||||||||||||||||||||20240115102500
DG1|1||I10|Essential (primary) hypertension|20240115|A
MSH|^~\\&|HIS|GH|EHR|GH|20240116090000||ADT^A01|MSG0002|P|2.5.1
PID|1||MRN-10002||TAN^AH KOW||19701130|M
PV1|1|O||ELEC|||||||||||||||VN-5002|||||||||||||||||||||||||20240116085500
DG1|1||E11.9|Type 2 diabetes mellitus without complications|20240116|A"""

In [3]:
def parse_hl7v2(message: str) -> dict[str, list[dict]]:
    """Parse pipe-delimited HL7 v2 (ER7) into segment dicts keyed for the
    Portiere hl7v2_2.5.1 standard (PID/PV1/DG1/OBX field names)."""
    FIELDS = {
        # segment -> {field_index: portiere_field_name}
        "PID": {3: "patient_id", 5: "patient_name", 7: "date_of_birth",
                8: "administrative_sex", 11: "patient_address"},
        "PV1": {2: "patient_class", 4: "admission_type", 19: "visit_number",
                44: "admit_date_time"},
        "DG1": {3: "diagnosis_code", 4: "diagnosis_description",
                5: "diagnosis_date_time", 6: "diagnosis_type"},
        "OBX": {2: "value_type", 3: "observation_identifier",
                5: "observation_value", 6: "units",
                14: "date_time_of_observation"},
    }
    out: dict[str, list[dict]] = {}
    for line in message.strip().splitlines():
        parts = line.strip().split("|")
        seg = parts[0]
        if seg not in FIELDS:
            continue
        record = {}
        for idx, name in FIELDS[seg].items():
            if idx < len(parts) and parts[idx]:
                record[name] = parts[idx]
        out.setdefault(seg, []).append(record)
    return out

In [4]:
# Split the feed into messages on MSH boundaries, then parse
raw_messages, current = [], []
for line in ADT_FEED.strip().splitlines():
    if line.startswith("MSH") and current:
        raw_messages.append("\n".join(current)); current = []
    current.append(line)
raw_messages.append("\n".join(current))

parsed = [parse_hl7v2(m) for m in raw_messages]
print(f"{len(parsed)} messages parsed")

2 messages parsed


## Hop 1 — HL7 v2 → FHIR R4

In [5]:
from portiere.local.cross_mapper import CrossStandardMapper

v2_to_fhir = CrossStandardMapper("hl7v2_2.5.1", "fhir_r4")

entity_map = v2_to_fhir.get_entity_map()

fhir_resources = []
for msg in parsed:
    for seg, records in msg.items():
        for rec in records:
            mapped = v2_to_fhir.map_record(seg, rec)
            fhir_resources.append({
                "resourceType": entity_map[seg],
                **{k: v for k, v in mapped.items() if v not in (None, "", {})},
            })

for r in fhir_resources:
    print(f"  {r.get('resourceType', '?'):12s} {json.dumps(r, default=str)[:84]}")

  Patient      {"resourceType": "Patient", "name": "DOE^JANE", "birthDate": "19850322", "gender": "
  Encounter    {"resourceType": "Encounter", "identifier": "VN-5001", "class": "IMP", "type": {"cod
  Condition    {"resourceType": "Condition", "code": "Essential (primary) hypertension", "onsetDate
  Patient      {"resourceType": "Patient", "name": "TAN^AH KOW", "birthDate": "19701130", "gender":
  Encounter    {"resourceType": "Encounter", "identifier": "VN-5002", "class": "AMB", "type": {"cod
  Condition    {"resourceType": "Condition", "code": "Type 2 diabetes mellitus without complication


## Hop 2 — FHIR R4 → OMOP CDM

Patient→person, Encounter→visit_occurrence, Condition→condition_occurrence.
The crossmap handles structural conversion; concept normalization (ICD-10 →
SNOMED standard concepts) is Stage-3 work — shown at the end.

In [6]:
fhir_to_omop = CrossStandardMapper("fhir_r4", "omop_cdm_v5.4")
print("entity map:", fhir_to_omop.get_entity_map())

omop_rows: dict[str, list[dict]] = {}
for r in fhir_resources:
    rtype = r.get("resourceType")
    if not rtype:
        continue
    mapped = fhir_to_omop.map_record(rtype, r)
    table = fhir_to_omop.get_entity_map().get(rtype, rtype)
    omop_rows.setdefault(table, []).append(mapped)

for table, rows in omop_rows.items():
    print(f"\n== {table} ({len(rows)} rows)")
    for row in rows:
        print("  ", json.dumps(row, default=str)[:100])

entity map: {'Patient': 'person', 'Encounter': 'visit_occurrence', 'Condition': 'condition_occurrence', 'MedicationRequest': 'drug_exposure', 'Observation': 'measurement', 'Procedure': 'procedure_occurrence', 'AllergyIntolerance': 'observation', 'DiagnosticReport': 'measurement'}

== person (2 rows)
   {"person_id": null, "gender_concept_id": 8532, "birth_datetime": "19850322", "person_source_value": 
   {"person_id": null, "gender_concept_id": 8507, "birth_datetime": "19701130", "person_source_value": 

== visit_occurrence (2 rows)
   {"visit_occurrence_id": null, "person_id": null, "visit_source_value": {"coding": [{"system": "http:
   {"visit_occurrence_id": null, "person_id": null, "visit_source_value": {"coding": [{"system": "http:

== condition_occurrence (2 rows)
   {"condition_occurrence_id": null, "person_id": null, "condition_source_value": "Essential (primary) 
   {"condition_occurrence_id": null, "person_id": null, "condition_source_value": "Type 2 diabetes mell


## Write OMOP tables

In [7]:
out = WORK / "omop_from_hl7v2"
out.mkdir()
for table, rows in omop_rows.items():
    df = pl.DataFrame([{k: str(v) for k, v in r.items()} for r in rows])
    df.write_csv(out / f"{table}.csv")
    print(f"  {table}.csv  ({df.height} rows, {df.width} cols)")

  person.csv  (2 rows, 4 cols)
  visit_occurrence.csv  (2 rows, 4 cols)
  condition_occurrence.csv  (2 rows, 4 cols)


## Concept normalization (Stage 3, offline)

The chained crossmaps carry the *source* codes through. To land standard
concept IDs, run the codes through the concept-mapping stage against your
vocabulary — here against the bundled offline subset.

In [8]:
# Offline configuration: BM25 retrieval over the bundled vocabulary subset,
# no embedding model (nothing to download), no reranker, and offline=True —
# construction FAILS if any provider could send data off this machine.
from portiere._demo_data import vocabulary_dir
from portiere.config import (
    EmbeddingConfig,
    KnowledgeLayerConfig,
    PortiereConfig,
    RerankerConfig,
)
from portiere.knowledge import build_knowledge_layer

knowledge_paths = build_knowledge_layer(
    athena_path=str(vocabulary_dir()),
    output_path=str(WORK / "knowledge_index"),
    backend="bm25s",
    vocabularies=["ICD10CM", "LOINC", "RxNorm"],
)

config = PortiereConfig(
    local_project_dir=WORK / "project",
    knowledge_layer=KnowledgeLayerConfig(backend="bm25s", **knowledge_paths),
    embedding=EmbeddingConfig(provider="none"),
    reranker=RerankerConfig(provider="none", model=""),
    offline=True,  # the executable no-egress guarantee (v0.4.0)
)
print("offline mode asserted:", config.offline)

2026-07-24 10:11:09 [info     ] athena.loading_concepts        path=/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/src/portiere/_demo_data/vocabulary/CONCEPT.csv


2026-07-24 10:11:09 [info     ] athena.concepts_loaded         count=27 vocabularies=['ICD10CM', 'LOINC', 'RxNorm']


2026-07-24 10:11:09 [info     ] athena.bm25s_built             count=27 path=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_qunsbv60/knowledge_index/concepts.json


2026-07-24 10:11:09 [info     ] athena.knowledge_layer_built   backend=bm25s concepts=27 paths={'bm25s_corpus_path': '/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_qunsbv60/knowledge_index/concepts.json'}


offline mode asserted: True


In [9]:
project = portiere.init(
    name="hl7v2-to-omop-concepts",
    target_model="omop_cdm_v5.4",
    vocabularies=["ICD10CM"],
    config=config,
)

dx_codes = [
    {"code": "I10", "description": "Essential (primary) hypertension", "count": 1},
    {"code": "E11.9", "description": "Type 2 diabetes mellitus without complications",
     "count": 1},
]
concept_map = project.map_concepts(codes=dx_codes)
for item in concept_map.items:
    print(f"  {item.source_code:8s} -> {str(item.target_concept_id):10s} "
          f"{(item.target_concept_name or '')[:44]} conf={item.confidence:.2f}")

2026-07-24 10:11:09 [info     ] PolarsEngine initialized      


2026-07-24 10:11:09 [debug    ] local_storage.initialized      base_dir=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_qunsbv60/project


2026-07-24 10:11:09 [info     ] local_storage.project_created  name=hl7v2-to-omop-concepts project_id=dde3195f-3403-43ef-b83f-40b341e9df3a


2026-07-24 10:11:09 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=bm25s


2026-07-24 10:11:09 [info     ] knowledge.creating_backend     backend=bm25s path=/var/folders/71/70h5b1l56xqdf5bywy0n43200000gn/T/portiere_nb_qunsbv60/knowledge_index/concepts.json


...

2026-07-24 10:11:09 [info     ] Stage 3 complete               auto_rate=100.0% total_codes=2


2026-07-24 10:11:09 [info     ] local_storage.concept_mapping_saved items_count=2 project=hl7v2-to-omop-concepts


2026-07-24 10:11:09 [info     ] project.concepts_mapped        auto_rate=100.0 project=hl7v2-to-omop-concepts total=2


2026-07-24 10:11:09 [warning  ] manifest.git_dirty             hint='Working tree has uncommitted changes; manifest records git_dirty=true' project_root=/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/docs/notebooks_examples


  I10      -> 1567980    Essential (primary) hypertension conf=4.05
  E11.9    -> 1567956    Type 2 diabetes mellitus without complicatio conf=5.24
